<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP project

In [1]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.8/443.8 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00


In [2]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch


In [3]:
# Load the model
chosen = 'llama'
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"},
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf"},
}

model = Llama.from_pretrained(repo_id=models[chosen]['repo_id'], # repository name
                            filename=models[chosen]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

In [4]:
rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/dominion.txt').text
rulebook[:100]

'# Dominion\nYou are a monarch, like your parents before you - a ruler of a small pleasant kingdom of '

In [5]:
# Check that input is inside context window
#tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

#tokens = tokenizer.encode(rulebook)
#print(len(tokens))

In [6]:
def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

In [7]:
import gc
def multiple_model_test(prompts, file_names, iterations, test_name):
    outputs = defaultdict(dict)

    for game,prompt_name,prompt,it in tqdm([(f,pn,p,it) for f in file_names for (pn,p) in prompts for it in range(iterations)]):
        rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
        out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)
        outputs[chosen+'-'+game+'-'+prompt_name][str(it)] = out['choices'][0]['message']['content']

    with open(f'{test_name}.out','w') as f:
        json.dump(dict(outputs),f)

    return outputs

# Rule extraction
1. Give the model a rulebook and prompt it to explain the game in simple, conversational terms to a child or other audiences. -> tree decomposition to test how well the model did
2. Test the ability of the model to find analogies of rules (?)
3. Test the ability to extract if-then rules (?)
4. Organize the rules of into a hierarchy: top-level objectives, mid-level phases, low-level actions. (?) (look into the paper)

In [8]:
prompts = [("kid", """You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation"""),
           ("analogies", """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the rules to a child by comparing it to something they already know (e.g. some other famous board games).
           se the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n"""),
]
game_names = [ 'dominion','7_wonders', 'catan', 'power_grid_recharged','ticket_to_ride',]

iterations = 1

multiple_model_test(prompts, game_names, iterations, 'extraction')

100%|██████████| 10/10 [03:59<00:00, 23.91s/it]


defaultdict(dict,
            {'llama-dominion-kid': {'0': 'Here\'s a summary of the game in plain language:\n\n**Goal of the game:** Your goal is to have the most points (called "victory points") at the end of the game. You get points by collecting certain cards, like Provinces and Gardens.\n\n**How a player wins:** The game ends when three piles of cards are empty or the Province pile is empty. Then, players count up their victory points, and the player with the most points wins.\n\n**What a turn looks like:**\n\n1. **Action phase:** You can play one Action card from your hand. This card can do special things, like gaining cards or trash cards.\n2. **Buy phase:** You can play Treasure cards from your hand to get coins, and then buy one card from the Supply using those coins.\n3. **Clean-up phase:** You discard all the cards you played and hand cards, and then draw a new hand of 5 cards.\n\n**Exceptions to standard rules:**\n\n* Some cards have special abilities, like gaining cards or

## Error detection

 Each rulebook is edited by inserting a set of 5 errors each of increasing difficulty:
 - level 1: **missing** -> an entire paragraph of the rulebook describing some core mechanic is missing
- level 2: **unsolvable** -> (in one line states you can draw two cards, in another one that you can draw only one)
- level 3: **incoherent** -> a mechanic that hardlocks the game (you cannot play train if you do not have train on the map but on another line clearly states you start with an empty map)
- level 4: **gamebreaking** -> a coherent but obviously gamebreaking mechanic (whenever you draw a card you can draw another card)
- level 5: **unbalanced** -> a coherent but very unbalanced mechanic (the first player can play two turns)

In [9]:
prompts = ["""You are an expert game board player.
Examine the rulebook provided by the user.
Proceed with a chain of thought:

- Scan the text linearly and note any statements that conflict with earlier ones.
- Look for gaps where a required mechanic is not explained.
- Check whether any mechanic could halt the game or give a player an overwhelming advantage.

Report the **most gamebreaking** problem you discover, quoting the relevant line and summarizing its impact.
If nothing stands out, reply “The rules appear consistent.”
"""]
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
iterations = 5

def multiple_model_test(prompts, file_names, iterations, test_name):
    outputs = defaultdict(dict)

    for game,prompt,it,lvl in tqdm([(f,p,it,lvl+1) for f in file_names for p in prompts for it in range(iterations) for lvl in range(5)]):
        rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/flawed_texts/lvl'+str(lvl)+'/'+game+'.txt').text
        out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)
        outputs[game][str(lvl)+str(it)] = out['choices'][0]['message']['content']

    with open(f'{test_name}.out','w') as f:
        json.dump(dict(outputs),f)

    return outputs

multiple_model_test( prompts, file_names, iterations, chosen+'_error_detection')

100%|██████████| 100/100 [30:44<00:00, 18.45s/it]


defaultdict(dict,
            {'llama-ticket_to_ride': {'10': 'The most gamebreaking problem I discover is related to the "Draw Train Cards" mechanic.\n\n**Problem:** A player can draw locomotive cards from the face-up deck, but they can also draw locomotives from the top of the deck (blind draw) and still draw a second card. This creates an inconsistent rule regarding locomotives and face-up drawing.\n\n**Relevant line:** "If you are lucky enough to get a locomotive from the top of the deck in a blind draw, it counts as a single card and you may still draw a total of 2 cards that turn."\n\n**Impact:** This rule allows a player to potentially draw multiple locomotives in a single turn, which can give them an overwhelming advantage in terms of route building and ticket completion. Locomotives are wild cards that can be part of any color set, so drawing multiple locomotives can greatly increase a player\'s chances of completing tickets and building long paths.\n\nThis problem can lead to

In [ ]:
# EVALUATION
sys_prompt = """You are a rule‑checking assistant.
Read ANSWER and GROUND TRUTH, narrate whether the explanations refer to the same rule, and conclude with a single word **EQUAL** or **DIFFERENT** on the final line.

### Examples
input:'''
ANSWER -> The Monopoly win condition states that the game ends when all other players have gone bankrupt,leaving a single player with all the assets. However, this rule can be problematic because it assumes that players will continue to take turns until bankruptcy occurs, which may never happen if a player repeatedly lands on “Free Parking” and collects cash without paying any fees. In such a scenario, the game could drag on indefinitely, making the win condition effectively unreachable. A more practical rule might impose a turn limit or a cash‑threshold to declare a winner when the game stalls.
GROUND TRUTH-> The problem is that the first player may reroll the dice any time.
'''

output:'''
The answer does not talk about rerolling the dice
DIFFERENT
'''

---

input:'''
ANSWER ->  After examining the rulebook, I've identified a **gamebreaking** problem: In the first phase of the game rules states that when a player draws a card any other player can choose to draw another card. This could create a scenario where all player are willing to draw until the deck is empty.
GROUND TRUTH -> infinite card draw at the start of the game.
Both the answer and the ground truth adress the card draw in first phases of the game as the problem
EQUAL
"""
verbose = True
res = []
for lvl in range(1,6):
    # construct the answer pair
    for answer, ground_truth in answer_pair:
        usr_prompt = f"ANSWER -> {answer}\nGROUND TRUTH -> {ground_truth}"
        out = model.create_chat_completion(generate_message(sys_prompt, usr_prompt), temperature=0.7)
        if(verbose):
            print(out['choices'][0]['message']['content'])
        res.append(['choices'][0]['message']['content'].splitlines()[-1])
    print(f"lvl{lvl}: {res.count('EQUAL')}/{len(res)}")


# Game classification
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration

In [ ]:
prompts = ["""test. For"""]
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
iterations = 1

multiple_model_test(prompts, file_names, iterations, 'classification')


In [ ]:
# EVALUATION
